# 02 — Feature Selection

This notebook covers predictive power analysis, multicollinearity filtering, and exploratory XGBoost-based feature selection for both PD and LGD models. Its output is the final feature lists and the dataframe checkpoint used by the EDA notebooks.

**Input:** `data/processed/checkpoint_end_Data_Diagnosis.pkl`  
**Output:** `data/processed/checkpoint_DataFrame_for_EDA.pkl`, `data/processed/features_PD_for_EDA.pkl`, `data/processed/features_LGD_for_EDA.pkl`

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import pickle
import os

from feature_selection import select_features, compute_plot_correlation_matrix, analyze_correlated_pairs, exploratory_feature_selection

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_end_Data_Diagnosis.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

print(f"Loaded. Shape: {df_master_features.shape}")

## 2. Define Feature Sets

PD and LGD models use different feature subsets. For LGD, `max_bal_owed` and `EAD` are excluded to prevent data leakage — `max_bal_owed` is the numerator of `Target_Variable_Loss`, and `EAD` is the denominator.

In [ ]:
# PD features: exclude LGD-specific columns and LGD target
features_PD = [
    col for col in df_master_features.columns
    if not col.endswith('_LGD') and col != 'Target_Variable_Loss'
]

# LGD features: exclude PD-specific columns, PD target, and leakage variables
features_LGD = [
    col for col in df_master_features.columns
    if not col.endswith('_PD')
    and col != 'Target_Variable_Default'
    and col != 'max_bal_owed'
    and col != 'EAD'
]

print(f"PD features: {len(features_PD)}")
print(f"LGD features: {len(features_LGD)}")

## 3. Predictive Power Analysis

Three metrics are computed per feature: mutual information, permutation importance (via a fast Random Forest), and Lasso coefficients. Features are retained under a conservative AND criterion: a feature must show signal in at least two of the three methods. Boruta is run for PD only.

In [ ]:
# PD: predictive power analysis with Boruta
report_PD = select_features(
    df_features=df_master_features[features_PD].drop(columns="Target_Variable_Default"),
    y=df_master_features["Target_Variable_Default"],
    run_boruta=True,
    sample_size=500_000
)

with open("data/processed/report_PD_Model.pkl", "wb") as f:
    pickle.dump(report_PD, f)

In [ ]:
# Load PD report (skip re-running)
with open("data/processed/report_PD_Model.pkl", "rb") as f:
    report_PD = pickle.load(f)

report_PD

In [ ]:
# LGD: Boruta skipped — sufficient signal from MI, permutation importance, and Lasso
report_LGD = select_features(
    df_features=df_master_features.loc[
        df_master_features["Target_Variable_Default"] == 1, features_LGD
    ].drop(columns="Target_Variable_Loss"),
    y=df_master_features.loc[
        df_master_features["Target_Variable_Default"] == 1, "Target_Variable_Loss"
    ],
    run_boruta=False,
    sample_size=100_000
)

with open("data/processed/report_LGD_Model.pkl", "wb") as f:
    pickle.dump(report_LGD, f)

In [ ]:
# Load LGD report (skip re-running)
with open("data/processed/report_LGD_Model.pkl", "rb") as f:
    report_LGD = pickle.load(f)

report_LGD

## 4. Correlation Analysis

A unified correlation matrix is computed using method-appropriate metrics:
- Numeric vs Numeric: Pearson correlation
- Categorical vs Categorical: Cramér's V
- Numeric vs Categorical: Eta squared

Pairs above threshold 0.75 are flagged. The feature with lower mutual information is discarded.

In [ ]:
# PD correlation matrix
corr_matrix_PD = compute_plot_correlation_matrix(df_master_features[features_PD])
pairs_PD, to_drop_PD = analyze_correlated_pairs(corr_matrix_PD, report=report_PD, threshold=0.75)
pairs_PD

In [ ]:
# EAD vs monthly_payment are highly correlated — monthly_payment captures debt burden more directly
# EAD is retained only for EL calculation, not as a predictive feature
features_PD = [col for col in features_PD if col not in to_drop_PD]

with open("data/processed/features_PD_AfterCorrelation.pkl", "wb") as f:
    pickle.dump(features_PD, f)

print(f"PD features after correlation filter: {len(features_PD)}")

In [ ]:
# LGD correlation matrix
corr_matrix_LGD = compute_plot_correlation_matrix(
    df_master_features.loc[df_master_features["Target_Variable_Default"] == 1, features_LGD]
)
pairs_LGD, to_drop_LGD = analyze_correlated_pairs(corr_matrix_LGD, report=report_LGD, threshold=0.75)
pairs_LGD

In [ ]:
features_LGD = [col for col in features_LGD if col not in to_drop_LGD]

with open("data/processed/features_LGD_AfterCorrelation.pkl", "wb") as f:
    pickle.dump(features_LGD, f)

print(f"LGD features after correlation filter: {len(features_LGD)}")

## 5. Exploratory XGBoost Feature Selection

An exploratory XGBoost model is trained to rank features by SHAP importance and determine a cutoff, avoiding deep EDA on features with negligible predictive contribution.

Two parallel cutoff criteria are computed: gain-based (fast reference) and SHAP-based (primary criterion, no cardinality bias). The SHAP cutoff uses 80% cumulative importance for PD and 90% for LGD — the higher threshold for LGD reflects the flatter SHAP curve caused by the noisier continuous target and compressed defaulted population.

### Leakage and survival bias exclusions

The following features are excluded from the exploratory model despite potentially appearing predictive:
- `princ_rec`, `interest_rec`, `remaining_princ_for_tot_amnt_fund`: post-default recovery variables — information only available after default occurs
- `has_late_fees_PD`: late fees accumulate after the default event
- `issue_date_year`, `issue_date_month`: survival bias — recent loans have had less time to default, creating a spurious negative correlation with default probability

In [ ]:
# PD exploratory model
leakage_vars_PD = [
    'princ_rec', 'interest_rec', 'remaining_princ_for_tot_amnt_fund',
    'has_late_fees_PD', 'issue_date_year', 'issue_date_month', 'Target_Variable_Default'
]
features_PD_model = [f for f in features_PD if f not in leakage_vars_PD]

importance_df_PD, selected_features_PD, metrics_PD = exploratory_feature_selection(
    df=df_master_features,
    features=features_PD_model,
    target='Target_Variable_Default',
    model_type='classification',
    sample_size=500_000
)

features_PD = selected_features_PD.copy()
print(f"Selected PD features: {len(features_PD)}")
print(features_PD)

In [ ]:
# Gain vs SHAP divergence analysis
cutoff_gain = metrics_PD['cutoffs']['gain_based']
cutoff_shap = metrics_PD['cutoffs']['shap_based']

gain_selected = importance_df_PD.sort_values('gain', ascending=False).head(cutoff_gain)['feature'].tolist()
shap_selected = importance_df_PD.sort_values('shap_mean_abs', ascending=False).head(cutoff_shap)['feature'].tolist()

print("In gain but not SHAP:", [f for f in gain_selected if f not in shap_selected])
print("In SHAP but not gain:", [f for f in shap_selected if f not in gain_selected])

In [ ]:
# LGD exploratory model — threshold raised to 0.90 (see documentation)
leakage_vars_LGD = [
    'princ_rec', 'interest_rec', 'remaining_princ_for_tot_amnt_fund',
    'has_late_fees_PD', 'issue_date_year', 'issue_date_month', 'Target_Variable_Loss'
]
features_LGD_model = [f for f in features_LGD if f not in leakage_vars_LGD]

importance_df_LGD, selected_features_LGD, metrics_LGD = exploratory_feature_selection(
    df=df_master_features[df_master_features["Target_Variable_Default"] == 1],
    features=features_LGD_model,
    target='Target_Variable_Loss',
    model_type='regression',
    sample_size=120_000,
    cumulative_gain_threshold=0.90
)

# dept_paym_income_ratio added via expert judgment — Basel core metric with independent signal
features_LGD = selected_features_LGD + ["dept_paym_income_ratio"]
print(f"Selected LGD features: {len(features_LGD)}")
print(features_LGD)

## 6. Checkpoint: Save Feature Lists and Dataframe

In [ ]:
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/checkpoint_DataFrame_for_EDA.pkl", "wb") as f:
    pickle.dump({"df_master_features": df_master_features}, f)

with open("data/processed/features_PD_for_EDA.pkl", "wb") as f:
    pickle.dump(features_PD, f)

with open("data/processed/features_LGD_for_EDA.pkl", "wb") as f:
    pickle.dump(features_LGD, f)

print(f"Saved. PD features: {len(features_PD)} | LGD features: {len(features_LGD)}")

In [ ]:
# Load checkpoints (skip re-running)
with open("data/processed/checkpoint_DataFrame_for_EDA.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

with open("data/processed/features_PD_for_EDA.pkl", "rb") as f:
    features_PD = pickle.load(f)

with open("data/processed/features_LGD_for_EDA.pkl", "rb") as f:
    features_LGD = pickle.load(f)